# gramlab — continue: longer fp16 training, BM25 retrieval, retrieval-conditioned Q&A, per-POS loss chart

Runs unattended via **Save Version → Save & Run All**. Settings: **GPU T4 x2** (or P100), **Internet on**.

**Before running:** *Add Input → Your Work → Notebooks →* the finished overnight notebook. Its output holds
`work/runs/*/ckpt.pt` and the token shards, which this notebook copies and continues from. If no previous output is
attached, everything is rebuilt from scratch (tiny baselines + `small_wiki` with a 3 h guard), so the job still finishes.

**Quota.** Kaggle gives ~30 GPU-hours a week (shared by P100 and T4 x2; a T4 x2 session counts once), 12 h per session,
20 GB of output. This job needs no interactive session: launch it with *Save & Run All* and close the tab (an open
interactive GPU session burns quota even when idle). Expected cost, full mode: **≈5–5.5 GPU-hours** (hard ceiling
≈ `TRAIN_HOURS` + 1.5 h). Quick mode: **≈0.75 GPU-hour**. Output ≈ 2 GB.

**Run `QUICK = True` once first** (next cell): it exercises every stage end-to-end in ~45 min — the torch paths were never
executed in the sandbox where this code was written, and a shallow bug is cheaper to find in a 45-minute run than at hour 3.

What happens (every long step is time-guarded):
1. re-fetch the Wikipedia text (the dumps were deleted by the overnight cleanup; the shards were kept) — 20 min
2. **continue the 25M `small_wiki` model in fp16** for `MORE_STEPS` (14000 ≈ 460M tokens ≈ 2 more epochs; the first run saw 0.7 at 3 s/step in emulated bf16) — schedule fitted to `TRAIN_HOURS`
3. **BM25 index** over ~330k Wikipedia lead paragraphs (pure numpy/scipy) — 5 min
4. **retrieval-conditioned Q&A**: 30k dialogues `Context: <passage> / Question: ...` → fine-tune `small_wiki_rag`; plus the
   plain `wiki_qa` fine-tune for a with/without-retrieval comparison — 15 min
5. **per-POS loss**: next-token loss by UPOS and by chunk position on gold GUM text for `tiny`, `tiny_upos`, `small_wiki` + chart, BLiMP — 20 min
6. sample answers with and without retrieval, a chat box, cleanup

## Parameters

In [ ]:
QUICK = False        # True = ~45-min smoke test of every stage (300 training steps, 5k Q&A dialogues, short evals). Run once before a full job.
TRAIN_HOURS = 4.0    # wall-clock budget for the continued pretraining in full mode (the whole job ≈ TRAIN_HOURS + 1.5 h at most)
MORE_STEPS = 14000   # steps to add to small_wiki in full mode: 14000 ≈ 460M tokens ≈ 2 more epochs of the 217M-token mix (-> ~3 in total).
                     # After 200 measured steps the trainer fits the cosine schedule into TRAIN_HOURS, so the lr still anneals if the GPU is slower.

## 0. Setup: code, corpora, previous output

In [ ]:
import os, subprocess, sys, time, glob, shutil, json
T0 = time.time()

def sh(cmd, must=True):
    print('\n$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if must and r.returncode != 0:
        raise SystemExit(f'command failed ({r.returncode}): {cmd}')
    return r.returncode

os.chdir('/kaggle/working')
if not os.path.exists('gramlab'):
    sh('git clone --depth 1 https://github.com/furevych/gramlab.git')
else:
    sh('git -C gramlab pull --ff-only', must=False)
os.chdir('/kaggle/working/gramlab')
os.makedirs('external', exist_ok=True); os.makedirs('results', exist_ok=True)
for url, d in [('https://github.com/gucorpling/amalgum', 'external/amalgum'),
               ('https://github.com/UniversalDependencies/UD_English-GUM', 'external/UD_English-GUM'),
               ('https://github.com/alexwarstadt/blimp', 'external/blimp')]:
    if not os.path.exists(d):
        sh(f'GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 {url} {d}')

# previous run's output, attached as an input: copy its work/ tree (shards + checkpoints, ~1-2 GB) and continue from it
prev = (sorted(glob.glob('/kaggle/input/**/work/runs/small_wiki/ckpt.pt', recursive=True))
        or sorted(glob.glob('/kaggle/input/**/work/runs/*/ckpt.pt', recursive=True)))
if prev:
    src = prev[0].split('/work/')[0] + '/work'
    print('previous output found:', src)
    if not os.path.exists('work'):
        shutil.copytree(src, 'work')
else:
    print('no previous output attached -> fresh start (tiny baselines + small_wiki from scratch)')
HAVE_SMALL = os.path.exists('work/runs/small_wiki/ckpt.pt')
HAVE_TINY = os.path.exists('work/runs/tiny/ckpt.pt') and os.path.exists('work/runs/tiny_upos/ckpt.pt')
print('checkpoints:', sorted(glob.glob('work/runs/*/ckpt.pt')))
import torch
from gramlab.amp import pick_precision
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '',
      '| training precision', pick_precision('cuda' if torch.cuda.is_available() else 'cpu')[4])

## 1. Wikipedia text again (for the BM25 index and the retrieval Q&A set) + any missing shards

In [ ]:
if not os.path.exists('data/simplewiki.jsonl'):
    sh('python scripts/fetch_wikipedia.py --lang simple --out data/simplewiki.jsonl')
NEED_EN_TEXT = not QUICK or not os.path.exists('work/bpe16000/shards/enwiki/meta.json')   # quick mode: index simplewiki only
if NEED_EN_TEXT and not os.path.exists('data/enwiki.jsonl'):
    sh(f'python scripts/fetch_wikipedia.py --lang en --max_articles {20000 if QUICK else 120000} --out data/enwiki.jsonl')
sh('python -m gramlab.prepare --config configs/small_wiki.json')      # tokenizer + shards; already-prepared corpora are skipped
WIKI = '--wiki data/simplewiki.jsonl' + (' --wiki data/enwiki.jsonl' if os.path.exists('data/enwiki.jsonl') else '')
for c in ['amalgum', 'simplewiki', 'enwiki', 'gum_test']:
    m = json.load(open(f'work/bpe16000/shards/{c}/meta.json')); print(f"{c:11} {m['n_words']:>12,} words -> {m['n_tokens']:>12,} tokens")
print(f'elapsed {(time.time()-T0)/60:.0f} min')

## 2. Tiny baselines (only when starting fresh): baseline vs grammar heads, ≈13 min each

In [ ]:
Q = ' --set train.steps=300 --set train.eval_every=100 --set train.eval_tokens=100000' if QUICK else ''
if not HAVE_TINY:
    sh('python -m gramlab.train --config configs/tiny.json' + Q)
    sh('python -m gramlab.train --config configs/tiny_upos.json' + Q)
    HAVE_TINY = True
else:
    print('tiny + tiny_upos checkpoints reused from the previous run')

## 3. Continue the 25M model in fp16 — `MORE_STEPS` more steps, re-warmed cosine from a lower peak lr, fitted into `TRAIN_HOURS`
`--resume` restores the optimizer and the fp16 loss scale; `--more_steps` re-spans the schedule to the new total; after 200 steps the trainer
measures s/step and, if fewer steps fit in `train.max_hours`, shortens the schedule so the learning rate still anneals before the guard.
(Fresh start: trains `small_wiki` from scratch under the same guard.)

In [ ]:
hours = 0.25 if QUICK else TRAIN_HOURS
if HAVE_SMALL:
    sh(f'python -m gramlab.train --config configs/small_wiki.json --resume --more_steps {300 if QUICK else MORE_STEPS} '
       f'--set train.lr=3e-4 --set train.max_hours={hours}' + (' --set train.eval_every=100 --set train.eval_tokens=100000' if QUICK else ''))
else:
    sh(f'python -m gramlab.train --config configs/small_wiki.json --set train.max_hours={hours}' + Q)
print('\neval history (work/runs/small_wiki/log.jsonl):')
for line in open('work/runs/small_wiki/log.jsonl'):
    r = json.loads(line); print({k: r[k] for k in ('step', 'train_lm_loss', 'eval_loss', 'eval_bits_per_byte', 'tokens_seen')})
print(f'elapsed {(time.time()-T0)/60:.0f} min')

## 4. BM25 index over the Wikipedia lead paragraphs (one passage per article, ≤100 words)

In [ ]:
sh(f'python -m gramlab.retrieval build {WIKI} --out work/bm25 --max_words 100')
for q in ['Tell me about Belgrade.', 'Who was Napoleon?', 'What is photosynthesis?', 'How do I bake bread?']:
    sh(f'python -m gramlab.retrieval search --index work/bm25 --q "{q}" -k 2', must=False)

## 5. Retrieval-conditioned Q&A → `small_wiki_rag`; plain Q&A → `small_wiki_sft` (for comparison)
`make_rag_qa.py` builds 30k dialogues: 55% the article's own passage → its opening sentences; 25% gold passage + a BM25 near-miss
in random order → the gold answer; 20% only a wrong passage → "I don't have information about X." The model learns to read, not to guess.

In [ ]:
sh(f'python scripts/make_rag_qa.py {WIKI} --index work/bm25 --out data/rag_qa.jsonl --n {5000 if QUICK else 30000}')
sh(f'python -m gramlab.finetune --init_from work/runs/small_wiki/ckpt.pt --corpus rag_qa --out small_wiki_rag --epochs {1 if QUICK else 2} --batch 32')
if not os.path.exists('data/wiki_qa.jsonl'):
    sh('python scripts/make_wiki_qa.py --amalgum external/amalgum/amalgum --out data/wiki_qa.jsonl')
sh(f'python -m gramlab.finetune --init_from work/runs/small_wiki/ckpt.pt --corpus wiki_qa --out small_wiki_sft --epochs {1 if QUICK else 3}', must=False)
print(f'elapsed {(time.time()-T0)/60:.0f} min')

## 6. Per-POS loss on gold GUM text (+ bits/byte, BLiMP) for `tiny`, `tiny_upos`, `small_wiki`
Where is a small model weak? Content words (NOUN/PROPN/NUM) are the facts it cannot know; function words (DET/ADP/AUX) are grammar.
If the grammar heads help, the gain should show up on function words and at chunk starts (B-*) more than inside chunks.

In [ ]:
runs = [r for r in ['tiny', 'tiny_upos', 'small_wiki'] if os.path.exists(f'work/runs/{r}/ckpt.pt')]
for r in runs:
    sh(f'python -m gramlab.evaluate --ckpt work/runs/{r}/ckpt.pt --corpus gum_test --per_pos --max_tokens {200000 if QUICK else 2000000} '
       f'--blimp external/blimp/data --blimp_limit {40 if QUICK else 300} --out results/eval_{r}.json > results/eval_{r}.log', must=False)
    if not os.path.exists(f'results/eval_{r}.json'):
        print(f'{r}: evaluation failed, see results/eval_{r}.log'); continue
    e = json.load(open(f'results/eval_{r}.json'))
    print(f"{r:11} bits/byte {e['eval_bits_per_byte']:.4f}  BLiMP {e.get('blimp', {}).get('_mean', float('nan')):.4f}  "
          f"tagged-token loss {e['tagged_loss']:.3f}  chunk start/inside/outside "
          + '/'.join(f"{e['per_chunk_position'][k]['loss']:.2f}" for k in ('B', 'I', 'O') if k in e['per_chunk_position']))
done = [r for r in runs if os.path.exists(f'results/eval_{r}.json')]
sh('python scripts/plot_pos_loss.py --results ' + ' '.join(f'results/eval_{r}.json' for r in done) + ' --out results/per_pos_loss.png')
from IPython.display import Image, display
display(Image('results/per_pos_loss.png'))

## 7. Same questions, with retrieval (`small_wiki_rag` + BM25) and without (`small_wiki_sft`)

In [ ]:
import sys, os, torch
sys.path.insert(0, '/kaggle/working/gramlab'); os.chdir('/kaggle/working/gramlab')
from gramlab.ckpt import load_checkpoint
from gramlab.tokenizer import USER, ASSISTANT, END, EOS
from gramlab.retrieval import BM25Index, format_rag_user
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
index = BM25Index.load('work/bm25')
models = {}
for name in ['small_wiki_rag', 'small_wiki_sft']:
    p = f'work/runs/{name}/ckpt.pt'
    if os.path.exists(p):
        models[name] = load_checkpoint(p, dev)
print('loaded', list(models), '| index', f'{len(index):,} passages')

def answer(name, user_text, temperature=0.5, top_k=40, top_p=0.9, rep=1.2, max_new=80):
    model, tok, state = models[name]
    ids = [USER] + tok.encode(user_text) + [END, ASSISTANT]
    x = torch.tensor([ids[-(model.ctx - 1):]], dtype=torch.long, device=dev)
    out = model.generate(x, max_new_tokens=max_new, temperature=temperature, top_k=top_k, top_p=top_p, repetition_penalty=rep, stop_ids=(END, EOS))
    return tok.decode(out[0, x.size(1):].tolist())

def ask(question, k=1, ctx_words=90, show=True):
    passages = index.retrieve(question, k=k, max_words=ctx_words)
    rag_user = format_rag_user(passages or [{'title': '(no match)', 'text': 'No passage found.'}], question)
    res = {}
    if 'small_wiki_rag' in models: res['with retrieval'] = answer('small_wiki_rag', rag_user)
    if 'small_wiki_sft' in models: res['no retrieval'] = answer('small_wiki_sft', question)
    if show:
        print('Q:', question)
        for p in passages: print(f"   retrieved [{p['score']:.1f}] {p['title']}: {p['text'][:140]}...")
        for k_, v in res.items(): print(f'   {k_:15}>', v)
        print()
    return res

PROMPTS = ['Tell me about Belgrade.', 'Tell me about Paris.', 'Who was Napoleon?', 'What is the capital of Japan?',
           'Tell me about the Danube.', 'How do I bake bread?', 'Who was Albert Einstein?', 'What is photosynthesis?',
           'What is a volcano?', 'Tell me about Zxqv Blorptown.']   # the last one has no article: the RAG model should say it has no information
for p in PROMPTS:
    ask(p)

## 8. Chat box (interactive sessions): retrieval-conditioned model, retrieved passage shown

In [ ]:
import ipywidgets as w
from IPython.display import display
box = w.Text(placeholder='ask about a place, a person, a thing...', layout=w.Layout(width='70%'), continuous_update=False)
btn = w.Button(description='Send'); out = w.Output()
def on_send(_=None):
    text = box.value.strip()
    if not text: return
    box.value = ''
    with out:
        ask(text)
btn.on_click(on_send)
box.observe(lambda ch: on_send() if ch['name'] == 'value' and ch['new'] else None, names='value')
display(w.HBox([box, btn]), out)

## 9. Keep the output small: drop the git clones and raw dumps (shards, checkpoints, BM25 index, Q&A sets and results stay)

In [ ]:
import shutil, os, time
for d in ['external', 'data/enwiki.jsonl', 'data/simplewiki.jsonl', '.git']:
    p = os.path.join('/kaggle/working/gramlab', d)
    if os.path.isdir(p): shutil.rmtree(p, ignore_errors=True)
    elif os.path.exists(p): os.remove(p)
os.system('du -sh /kaggle/working/gramlab/work/* /kaggle/working/gramlab/results /kaggle/working/gramlab/data 2>/dev/null')
print(f'total run time {(time.time()-T0)/3600:.2f} h')